In [ ]:
import geopops
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import starsim as ss
import warnings
warnings.resetwarnings()  # optional but helps in notebooks
warnings.filterwarnings(
    "ignore",
    category=RuntimeWarning,
    module=r"starsim\.utils",
)

# 1.0 Run Geopops

**GeoPops** is a package for generating geographically and demographically realistic synthetic populations for any US Census location using publically available data. Population generation includes three steps:
1. Generate individuals and households using combinatorial optimization (CO)
2. Assign individuals to school and workplace locations using enrollment data and commute flows
3. Connect individuals within locations using graph algorithms

Resulting files include a list of agents with attributes (e.g., age, gender, race/ethnicity) and networks detailing their connections within home, school, workplace, and group quarters (e.g., correctional facilities, nursing homes) locations. GeoPops is meant to produce reasonable approximations of state and county population characteristics with granularity down to the Census Block Group (CBG).   GeoPops builds on a previous repository, [GREASYPOP-CO](https://github.com/CDDEP-DC/GREASYPOP-CO/tree/main) (One Health Trust), and incorporates the following changes:
- All code wrapped in convenient Python package that can be pip installed
- Compatibility with Census data beyond 2019
- Automated data downloading
- Users can adjust all config parameters from the front-end
- Class for exporting files compatible with the agent-based modeling software [Starsim](https://starsim.org/) (Institute for Disease Modeling)

## 1.1 Obtain Census API
First, obtain a **Census API key** [here](https://api.census.gov/data/key_signup.html), which will be used for pulling Census data. Getting and activating a key may take a few mintues.

## 1.2 Create Python environment
You'll also need a **Python environment** with the dependencies listed in the GeoPops `pyproject.toml`. We suggest making a virtual environment different from you base environment using python 3.12 or 3.13 and storing it locally so that you never have to re-download package files from the Cloud. Create a local folder where you want to store your GeoPops files. Download the `pyproject.toml` file from this repo and put it in your selected folder. In your terminal, navigate to that folder and try the following commands (for Mac). Don't copy the comments; these are just for reference.

    cd "YOUR_FOLDER"                # Navigate to your folder
    python3.12 -m venv geopops-env  # Create a virtual environment with python 3.12 or 3.13
    source geopops-env/bin/activate # Activate the virtual environment
    pip install geopops             # Install geopops and dependencies
    pip install ipykernel           # Install ipykernel so you can find the environment in Jupyter notebooks
    python -m ipykernel install --user --name geopops-env --display-name "Python (geopops-env)"



## 1.3 Quick Use Guide
You can generate a GeoPops population and resulting output files by creating a dictionary of parameters and passing it into the command `geopops.RunAll()`. The following cell creates a population of Spartanburg County, SC. It takes about 8-15 min to run. The remaining notebook sections go into detail on GeoPops package structure, data sources, methodology, and output.

In [ ]:
# Define parameters
pars_geopops = {'path': "YOUR_OUTPUT_FOLDER", # Set a folder where you want outut files to be stored
                'census_api_key': "YOUR_API_KEY", # Your Census API key
                'main_year': 2019, # Year of data
                'geos': ["45083"], # State or county fips of your geographical location of interest.Example of Spartanburg SC
                'commute_states': ["45","37"], # State fips of commute data to download. Example of SC, NC
                'use_pums': ["45","37"], # State fips of PUMS data to download. Example of SC and NC
                } 

# Generate population with .RunAll()
geopops.RunAll(pars=pars_geopops)

## 1.4 GeoPops package structure
GeoPops is structured using python classes and methods. Classes are listed below, and methods are explained in subsequent notebook sections.

| Class | Description |
| -------- | -------- | 
| geopops.RunAll() | Takes parameter dictionary as input. Automatically runs all the following classes sequencially |
| geopops.WriteConfig() | Takes parameter dictionary as input. Writes config.json with specified parameters |
| geopops.DownloadData() | Downloads raw data files from Census API and URLs |
| geopops.ProcessData() | Processes data for population generation |
| geopops.GeneratePop() | Builds the population and networks |
| geopops.ForStarsim() | Exports agent and network files for Starsim |

All the parameters to customize your population are stored a file called `config.json`. There are a lot of parameters you can adjust (see `data/config_parameters.csv` for full list). 

| Parameter | Definition |
| -------- | -------- | 
| path | Chose a folder where you want output files to be stored |
| census_api_key | Your Census API key |
| main_year | Year of your population, corresponds to year of Census data. main_year=2019 means school year 2019/2020 |
| geos | State or county FIPs codes of your main geography |
| commute_states | State FIPs codes where people commute to and from work (The populaton is "closed" except for commuters) |
| use_pums | State FIPS codes of the PUMS households sampled for combinatorial optimization |

The first step is defining these parameters in a dictionary. Then you can pass this dictionary into `RunAll()` (section 1.3) or into `WriteConfig()`. The `WriteConfig.get_pars()` method prints out the full config.json parameter dictionary.

In [2]:
# Define parameters
pars_geopops = {'path': "data", # Set a folder where you want outut files to be stored
                'census_api_key': "4ab35c539d0d99127684ce3a387231ee78612a46", # Your Census API key
                'main_year': 2019, # Year of data
                'geos': ["45083"], # State or county fips of your geographical location of interest.Example of Spartanburg SC
                'commute_states': ["45","37"], # State fips of commute data to download. Example of SC, NC
                'use_pums': ["45","37"], # State fips of PUMS data to download. Example of SC and NC
                } 

# Define config.json with parameter dictionary
c = geopops.WriteConfig(pars=pars_geopops) # Define parameters for pop generation in config.json

# Method for viewing all parameters from config.json
c.get_pars() # View all parameters from config.json


Running WriteConfig()
-- Updated config.json with parameter dictionary
{
  "path": "data",
  "census_api_key": "4ab35c539d0d99127684ce3a387231ee78612a46",
  "julia_env_path": "YOUR_JULIA_ENV_PATH",
  "main_year": 2019,
  "geos": [
    "45083"
  ],
  "commute_states": [
    "45",
    "37"
  ],
  "use_pums": [
    "45",
    "37"
  ],
  "decennial_year": 2010,
  "acs_required": [
    "B01001",
    "B09019",
    "B09020",
    "C24030",
    "B23025",
    "C24010",
    "B11016",
    "B11012",
    "B23009",
    "B11004",
    "B19001",
    "B22010",
    "B09021",
    "B09018",
    "B11001H",
    "B11001I",
    "B25006"
  ],
  "dec_required": [
    "P43",
    "P18"
  ],
  "inc_adj": 1.010145,
  "inc_cats": [
    "q1_1",
    "q1_2",
    "q1_3",
    "q2",
    "q3",
    "q4",
    "q5"
  ],
  "inc_cols": [
    [
      "Less than $10,000"
    ],
    [
      "$10,000 to $14,999",
      "$15,000 to $19,999",
      "$20,000 to $24,999"
    ],
    [
      "$25,000 to $29,999",
      "$30,000 to $34,999

## 1.5 Download raw data
The `DownloadData()` class pulls raw data files from the internet using the Census API and urls strings. Output files go into the folders census, geo, pums, school, and work. These folders will show up in the output directory you chose as the data are downloaded. `folders_files_tree.rtf` shows how all the files are stored in your output directory. Data are pulled from the following sources:

| Data Source | Role in GeoPops |
|---|---|
| [American Community Survey (ACS)](https://www.census.gov/programs-surveys/acs) | Provides block-group-level demographic and socioeconomic distributions (e.g., age, households, income) that GeoPops matches when constructing the synthetic population. |
| [Decennial Census](https://www.census.gov/programs-surveys/decennial-census) | Used to construct group quarters compositions (institutional vs non‑institutional, civilian vs military) |
| [Public Use Microdata Sample (PUMS)](https://www.census.gov/programs-surveys/acs/microdata.html) | Provides person- and household-level microdata that are reweighted and combined (via combinatorial optimization) to create realistic synthetic households and individuals. |
| [TIGER/Line Shapefiles](https://www.census.gov/geographies/mapping-files/time-series/geo/tiger-line-file.html) | Provide the geographic boundaries (tracts, block groups, etc.) needed to locate agents spatially and to link Census counts to geography. |
| [LEHD LODES (Longitudinal Employer–Household Dynamics)](https://lehd.ces.census.gov/data/#lodes) | Provides origin-destination commute patterns and workplace locations, used to assign workers to jobs and build the work contact network. |
| [County Business Patterns (CBP)](https://www.census.gov/programs-surveys/cbp.html) | Supplies employment counts by industry and geography, used to characterize and weight workplaces in the synthetic population and work network. |
| [Census Tract-to-PUMA Crosswalk](https://www.census.gov/geographies/reference-files/time-series/geo/relationship-files.html) | Links Census tracts to PUMAs so that PUMS microdata (available at the PUMA level) can be aligned with tract/block-group targets. |
| [MCDC Geocorr (2018/2022)](https://mcdc.missouri.edu/applications/geocorr.html) | Provides crosswalks between tracts/CBGs and higher-level geographies (counties, CBSAs, urban-rural), used to attach regional attributes and to aggregate or filter results. |
| [NCES EDGE School Data](https://nces.ed.gov/programs/edge) | Supplies public school locations and characteristics, used to place students and staff and to build realistic school contact networks. |


### Data caveats
GeoPops uses the most recent data available from each source, but the most recent year may not be consistent across all data sources. The following table describes the impact of data year inconsistancies on the synthesized population. 

| Data source | Availability | Impact on GeoPops |
|---|---|---|
| ACS | If `main_year >= 2023`, tables `B09019`, `B09020`, and `B09021` fall back to 2022 because newer releases are not available via Census API | Most targets still use requested-year ACS where available, but some household/GQ and age-structure constraints become mixed-year. This can affect employment/GQ balancing and age composition. |
| PUMS | Available through 2023. Password required from Census website for years >=2024 | If `main_year > 2023`, PUMS falls back to 2023 microdata. Household/person samples come from 2023 structure and are reweighted to target-year ACS marginals. |
| LODES | Available through 2023. Not yet published for years >=2024 | If `main_year > 2023`, commute and workplace assignment fall back to 2023 origin-destination flows, so work network patterns reflect 2023 geography/economy. |
| Geocorr | Uses `geocorr2018` for pre-2020 runs; `geocorr2022` for 2020+ runs | Determines CBG↔county/cbsa/urban-rural mapping. Mixed-geography years (2020-2022) can increase missing geo links and reduce fine-scale geographic fidelity. |
| NCES | Available through 2024 | If `main_year > 2024`, school location/enrollment/staff data fall back to the latest available year, so school assignment/network reflect fallback-year school structure. |

**Caveat take-away:** For any of the mixed-year or mixed-geography data combinations, combinatorial optimization (section 1.7.1 in this notebook) will have a harder time selecting a set of PUMS households that closely matches the ACS joint distributions, especially at finer geographic scales like CBG.

Each data source has its own method in the `DownloadData()` class. In the cell below, set `auto_run=True` to download all data at once, or `auto_run=False` to run one method at a time. Runtime is about 4 min on a 2026 Macbook Pro. Downloading school data especially takes a while so be patient. You may want set your computer not to go to sleep in case this breaks the run.

In [ ]:
# Create DownloadData object
d = geopops.DownloadData(auto_run=True, verbose=1) # verbose=1 prints output statements

# Can set auto_run=False to run one method/data source at a time
# d.pull_census_data()
# d.pull_pums_data()
# d.download_shapefiles()
# d.pull_LODES()
# d.download_ct_puma_crosswalk()
# d.geocorr_files()
# d.download_cbp_data()
# d.download_school_data() # This one can take >15 minutes 


Running DownloadData.download_shapefiles()
-- Downloading from https://www2.census.gov/geo/tiger/TIGER2019/BG/
-- geo/tl_2019_45_bg.zip

Running DownloadData.pull_LODES()
-- Downloading from https://lehd.ces.census.gov/data/lodes/
-- work/sc_od_main_JT01_2019.csv.gz
-- work/sc_od_aux_JT01_2019.csv.gz
-- work/sc_wac_S000_JT01_2019.csv.gz
-- work/nc_od_aux_JT01_2019.csv.gz

Running DownloadData.download_ct_puma_crosswalk()
-- Downloading from https://www2.census.gov/geo/docs/maps-data/data/rel/
-- geo/2010_Census_Tract_to_2010_PUMA.txt

Running DownloadData.geocorr_files()
-- Copying package files from src/geopops/geocorr
-- geo/geocorr2018*

Running DownloadData.download_cbp_data()
-- Downloading from https://www2.census.gov/programs-surveys/cbp/datasets/2016/
-- work/cbp16co.zip

Running DownloadData.download_school_data()
-- Downloading from https://nces.ed.gov/programs/edge/data/
-- school/EDGE_GEOCODE_PUBLICSCH_1920.zip


## 1.6 Process data
The `ProcessData()` class reformats all the raw data into interim files that will be used in combinatorial optimization, school and workplace assignment, and network generation. Intirem files go into the folder processed. Runtime is about 1 minute with a 2026 MacBook Pro. Set `auto_run=True` to run all at once or `auto_run=False` to run one processing step at a time. If running one at a time, must be done in order because each method depends on the output of the method function.

In [2]:
p = geopops.ProcessData(auto_run=False, verbose=1) 

# Can run one at a time with auto_run=False but must be in order
# because each function depends on the output of the previous one
# p = geopops.ProcessData(auto_run=True) # auto_run=True to run all 
# p.generate_samples()
# p.generate_gq()
# p.generate_targets()
# p.calc_commute_marginals()
# p.generate_work_sizes()
# p.generate_schools() # Takes a long time

## 1.6.1 Quality Check
For 2020–2022, some Census products use mixed geography vintages, so small-area targets and PUMA-based microdata are not always perfectly aligned. In practice, this can create partial puma mismatches and force optimization to rely more on county/cbsa/urban-rural fallback instead of local PUMA matching. The data are still usable, but these years should be interpreted with more caution for fine geographic fit. This will make more sense after reading section **1.7.1 Combinatorial optimization** and the **1.9 More on CO()** section at the end of this notebook. 

After running `ProcessData`, you can run the `ProcessData.quality_check()` method to check that the processed geography files are ready for optimization. The check confirms that all target CBGs’ state–PUMA codes (st_puma) are fully covered by the PUMS sample pool, and it reports any missingness in key geography fields (st_puma, county, cbsa, rural R, and urban U) on either the sample side (samp_geo.csv) or the target CBG side (cbg_geo.csv). A final co_readiness_summary status of good indicates that the census/geo inputs are clean and well-aligned, so subsequent population synthesis steps can proceed with confidence.

**Take-away:**
Years <=2019 and year 2023 produce more robust populations at fine geographic scale (e.g., CBG). Years 2020-2022 can still produce reasonable approximations for aggregate regions (e.g., county or state) but should be treated with caution when looking at finer geographic scales in spatial analyses.

In [4]:
p.quality_check();


*** Running ProcessData.quality_check() ***

*** Running QualityCheck.print_results() ***
QualityCheck results
data_dir: data

st_puma_overlap
  target_unique_pumas: 2
  sample_unique_pumas: 108
  covered_target_pumas: 2
  overlap_pct: 1.000

samp_geo_missingness
  st_puma: missing_n=0, missing_pct=0.000
  county: missing_n=0, missing_pct=0.000
  cbsa: missing_n=0, missing_pct=0.000
  R: missing_n=0, missing_pct=0.000
  U: missing_n=0, missing_pct=0.000

target_geo_missingness
  st_puma: missing_n=0, missing_pct=0.000
  county: missing_n=0, missing_pct=0.000
  cbsa: missing_n=0, missing_pct=0.000
  R: missing_n=0, missing_pct=0.000
  U: missing_n=0, missing_pct=0.000

co_readiness_summary
  status: good


## 1.7 GeneratePop
The `GeneratePop()` class consists of three methods, which can be run separately or all together (sequentially) using `auto_run=True`.For a more detailed explanation of the methods for each step, see [Tulchinsky et al. 2026](https://pubmed.ncbi.nlm.nih.gov/41762550/).
1. `.CO()` - Combinatorial Optimization
2. `.SynthPop()` - School and workplace assignment and network generation
3. `.Export()` - Export outputs to csv

## 1.7.1 Combinatorial optimization
GeoPops uses combinatorial optimization to sample households from the Public Use Microdata Samples (PUMS) until a set is found that reasonably matches overall population characteristics from the American Comunity Survey (ACS) for your geography of interest (defined in geos in `WriteConfig()`).

**PUMS:** Anonymized subset of Census survey responses for regions of around 100,000 people. Contains compositional details of households and members (e.g., the number of five-member households with school-age children).

**ACS:** Geography and demographics for Census areas down to CBG level. Restricted to select summaries of marginal counts (e.g., how many households in a locale have five members and the number of school-age children). Missing household compositional details. 

The distributions of demographic characteristics in a PUMS area may not be representative of the overall population area. CO is a way to combine a set of households from PUMS so that the joint distributions for demographic characteristics (age, income, gender, etc) of all the individual agents within that set reasonably match ACS data.

See the section **More detail on CO()** at the end of this notebook for further explaination of how `CO()` works and a description of printouts when running.

Setting the `random_seed` is a way to produce the same population across multiple runs. 

Runtime for `CO()` for this example of Spartanburg County is about 2 minutes on a 2026 MacBook Pro.

In [3]:
g = geopops.GeneratePop(run_all=False, random_seed=123)
g.CO()

*** Running GeneratePop.CO() ***

County 45083: 195 CBGs

Optimizing 195 CBGs at PUMA level
-- After PUMA pass; 1 above threshold (will rerun); E0 min/mean/max: 14.31 / 14.92 / 17.75
Optimizing 1 CBG(s) at county level
-- After county pass: 1 still above threshold; E0 min/mean/max: 14.31 / 14.91 / 16.56
Optimizing 1 CBG(s) at CBSA level
-- After CBSA pass: 1 still above threshold; E0 min/mean/max: 14.31 / 14.91 / 16.56
Optimizing 1 CBG(s) at urbanization level
-- After urbanization pass: 0 still above threshold; E0 min/mean/max: 14.31 / 14.91 / 15.00

195/195 CBGs met the stopping criterion (E0 <= 15.0) for the Freeman-Tukey distance score.


## 1.7.2 Assignment and network generation
The `RunJulia.SynthPop()` method performs school and workplace assignment and network generation. Output files go into the folder pop_export. Networks are stored as upper adjacency matrices in .mtx format. 

**Assignment**
| Locations | Methods | Data | Sources |
| -------- | -------- | -------- | -------- |
| Schools | Students assigned to closest school that offer required grade. Teachers assigned to school by selecting teachers from nearby CBGs | Student and staff counts for each grade level in public schools | National Center for Education Statistics (NCES) |
| Workplaces | Iterative proportional fitting (IPF) to make origin-destination commute matrix. IPF to make industry by destination commute matrix for each origin CBG. Includes agents who live in the geo area but travel outside for work | Origin-destination and workplace category matrices | Longitudinal Employer-Household Dynamics (LEHD) |

**Network generation** 
| Network | Algorithm | Parameters |
| -------- | -------- | -------- |
| Household | All agents within a household are fully connected |  |
| School | Stochastic Block Model within each school | Associativity ⍺=0.9. Mean degree K=12. Blocks are grade levels (Pre-k – 12) |
| Workplace | Stochastic Block Model within each workplace | Associativity ⍺=0.9. Mean degree K=8. Blocks are income levels (<40k or ≥ 40k) |
| Group quarters (GQ) | Watts-Strogatz Model within each GQ facility | Mean degree K=12. Rewiring probability β=0.25 |

The Stochastic Block Models are connecting agents within schools so that each agent has a mean of 12 contacts, and ~90% of these contacts are within the same grade. You can adjust the network generation algorithms by changing the following parameters with `WriteConfig.py`:

| Network parameters | Definition | Default value |
| -------- | -------- | -------- |
| school_associativity_coefficient | School associativity | 0.9 |
| school_K | Mean degree in school network | 8 |
| work_associativity_coefficient | Workplace associativity | 0.9 |
| workplace_K | Mean dgree in workplace network | 12|
| gq_K | Mean degree in group quarters network | 12 |
| netw_B | rewiring probability for small-world GQ network | 0.25 |

Runtime for `SynthPop()` is about 1 minute. `SynthPop()` needs items in memory from `CO()` so if you're jumping into a restarted notebook, you'll need to rerun `CO()`.

In [7]:
g.SynthPop()


*** Running GeneratePop.SynthPop() ***

Generating people
-- County 45083: 297703 people, 116645 households
-- Total: 297703 people, 116645 households

Generating group quarters
-- County 45083: 7057 group quarters residents, 31 group quarters
-- Total: 7057 group quarters residents, 31 group quarters

Generating schools
-- County 45083: 55362 students, 74 schools
-- Total: 55362 students, 74 schools
-- 0 schools assigned 0 students

Generating workplaces
-- Generating OD matrices, exporting interim files
-- processed/od_rows_origins.csv
-- processed/od_columns_dests.csv
-- processed/od_AGR_EXT.csv.gz
-- processed/od_CON.csv.gz
-- processed/od_MFG.csv.gz
-- processed/od_WHL.csv.gz
-- processed/od_RET.csv.gz
-- processed/od_TRN_UTL.csv.gz
-- processed/od_INF.csv.gz
-- processed/od_FIN.csv.gz
-- processed/od_PRF.csv.gz
-- processed/od_EDU.csv.gz
-- processed/od_MED.csv.gz
-- processed/od_ENT_art.csv.gz
-- processed/od_ENT_food.csv.gz
-- processed/od_SRV.csv.gz
-- processed/od_ADM_MIL.cs

## 1.7.3 Export files
The `Export()` class exports output as csv and mtx files into the folder pop_export. Runtime is about 2 seconds. `Export()` requires objects in memory from `CO()` and `SynthPop()` so you need to run them sequentially if running one at a time.

| File| Description |
| -------- | -------- | 
| people.csv | List of agents with attributes who live inside geo area |
| hh.csv | List of of households with location, PUMs sample index, and household size |
| cbg_idxs.csv | Maps GeoPops cbg_id to 15-digit Census CBG geocode |
| sch_students.csv | List of students by school code |
| sch_workers.csv | List of school workers by school code |
| company_workers.csv | List of agents by employer locations inside geo area, type, and employer id  |
| outside_workers.csv | List of agents that work outside the geo area |
| gq_residents.csv | List of GQ residents by GQ location and GQ id |
| gq_workers.csv | List of GQ workers by location and GQ id |
| gqs.csv | List of GQ facilities with location, type, and GQ id |
| adj_mat_keys.csv | Full list of agents but no attributes |
| adj_dummy_keys.csv | List of agents who live outside geo area and commute in for work, no attributes. Inluded in workplace network |
| adj_out_workers.csv | List of agents who live in area and commute out to work, no attributes. Not in workplace network |
| adj_upper_trang_hh.mtx| Household network upper adjacency matrix |
| adj_upper_trang_sch.mtx| School network upper adjacency matrix |
| adj_upper_trang_wp.mtx| Workplace network upper adjacency matrix |
| adj_upper_trang_gq.mtx| Group quarters network upper adjacency matrix |
| adj_upper_trang_non_hh.mtx| Combination of School, workplace, and gq (NOT "Other" or "Community" network) |

In [8]:
g.Export()


*** Running GeneratePop.Export() ***

Exporting cbg_id to cbg_geocode crosswalk
-- pop_export/cbg_idxs.csv

Exporting people and households
-- pop_export/hh.csv
-- pop_export/people.csv

Exporting school data
-- pop_export/sch_students.csv
-- pop_export/sch_workers.csv

Exporting group quarters data
-- pop_export/gqs.csv
-- pop_export/gq_residents.csv
-- pop_export/gq_workers.csv

Exporting workplace data
-- pop_export/company_workers.csv
-- pop_export/outside_workers.csv

Exporting network data
-- pop_export/adj_upper_triang_hh.mtx
-- pop_export/adj_upper_triang_non_hh.mtx
-- pop_export/adj_upper_triang_wp.mtx
-- pop_export/adj_upper_triang_sch.mtx
-- pop_export/adj_upper_triang_gq.mtx
-- pop_export/adj_mat_keys.csv
-- pop_export/adj_dummy_keys.csv
-- pop_export/adj_out_workers.csv


## 1.8 Format for Starsim
Starsim ([Kerr et al. 2024](https://starsim.org/)) is an open-source agent-based modeling software by the Institute for Disease Modeling. It is module-based, allowing the user to customize the disease, contact structure, and population.

The `geopops.ForStarsim()` class has three methods for formatting a GeoPops population for Starsim.
* `.People()` creates a Starsim people object (`ppl.pkg`) of agents and attributes. Also creates `people_all.csv` which lists all agents and attributes including dummy agents who live outside the geo area.
* `.GPNetwork()` transforms the upper adjacency matrices into a Starsim Network. Stored as `pop_export/starsim/net_w.csv`, etc.
* `.SubgroupTracking()` returns a dataframe of outcomes overtime by subgroup. We'll use this later in `5_subgroup_tracking.ipynb`

Files are exported into the the following folders and will be loaded into a Starsim simulation in the other notebooks in this tutorial.
| Folder/File | Description |
| -------- | -------- | 
| pop_export/people_all.csv | List of all agents with attributes, including dummy agents who live outside geo area |
| pop_export/starsim/ppl.pkl | Starsim people object of your GeoPops population |
| pop_export/starsim/net_g.csv | Grouq quarters network as edge list |
| pop_export/starsim/net_h.csv | Home network as edge list |
| pop_export/starsim/net_s.csv | School network as edge list |
| pop_export/starsim/net_w.csv | Work network as edge list |

## 1.8.1 People
Create Starsim people object from your GeoPops agents and attributes and view it as a dataframe. The People object is explained in more detail in `2_explore_people.ipynb`.

In [2]:
ppl = geopops.ForStarsim.People()


*** Running ForStarsim.People() ***
Initializing sim with 357963 agents
Starsim People object created and saved successfully


## 1.8.2 Make networks
`ForStarsim.GPNetwork()` reformats the upper adjacency matrices (mtx files) for each network into dataframes (also saved as csv files), where `p1` (person 1) is connected to `p2` (person 2) and `edge_weight` is the weight of their connection. The link between the two agents is bi-directional so if either is infected, they could infect the other. With Starsim, edge_weight gets turned into a multiplier in the transmission equation, so increasing edge_weight in one network relative to others would increase transmissions in that network relative to others. More on networks in `3_explore_networks.ipynb`.

In [3]:
h = geopops.ForStarsim.GPNetwork(name='homenet', edge_weight=1.0)
s = geopops.ForStarsim.GPNetwork(name='schoolnet', edge_weight=1.0)
w = geopops.ForStarsim.GPNetwork(name='worknet', edge_weight=1.0)
g = geopops.ForStarsim.GPNetwork(name='gqnet', edge_weight=1.0)


*** Running ForStarsim.GPNetwork._create_networks() ***
Network csv files created and saved successfully


## 1.9 More detail on CO()

CO iteratively adjusts the set of PUMS microdata samples within each geography so that aggregate totals (age, households, etc.) match joint distributions from the ACS data. Each pass focuses on a different geographic level, starting with PUMA (Public use Microdata Area). If a CBG still fails the tolerance criteria after a pass, it is flagged and re‑optimized in subsequent passes at bigger geographic levels (first county, then Core Based Statistical Area (CBSA), then urbanization (urban/rural)) until all geographies fall within the acceptable error threshold or no further improvement is possible. With `WriteConfig()` you can change the error threshold and the number of generations the optimizer runs by redefining the parameters `CO_crit_val` and `CO_maxgens`, which have default values of 15.0 and 200,000, respectively. The printout includes the current value of the error metric (E0) for that optimization run (how far you are, on average, from the target constraints using the Freeman-Tukey distance score).


If a large number of CBGs must be rerun at each geography level and only finally resolve at the urban/rural step, it means:
* The constraints are hard to satisfy at fine spatial scales (PUMA, state, county, CBSA) given the available microdata; many CBGs can’t match their detailed target distributions without violating others.
* When you move to coarser groupings (urban/rural), the algorithm can trade off errors across CBGs within the same urban/rural class, so it becomes easier to find a configuration where all units fall below the error threshold.
* Practically, this indicates that your population still matches ACS targets well in aggregate, but individual CBGs are being heavily adjusted and some local estimates are more model‑driven than data‑driven.